<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🗺️ 定制 Agent 行为的三种方式

### Wrap 风格 Middleware vs Node 风格 Middleware vs 工具调用

**一句话定位**：三种机制长得像，但作用的层面和"谁说了算"完全不同。

</div>

![Middleware Comparison Table](resources/middleware_comparison_table.svg)

LangChain 给了三种截然不同的机制来定制 agent 运行时的行为：

| | 🔀 Wrap 风格 Middleware | 🪝 Node 风格 Middleware | 🔧 工具调用 |
|---|---|---|---|
| **接收** | `ModelRequest` | `State` + `Runtime` | `ToolRuntime` |
| **谁控制** | 你（开发者） | 你（开发者） | LLM（agent 自己） |
| **调整什么** | 工具、提示词、模型 | state（消息、自定义字段） | state 或读取上下文 |
| **何时跑** | 每次 LLM/工具调用前后 | agent 或 tool 节点前后 | LLM 决定要调用工具时 |
| **例子** | 给内部用户额外工具 | 裁剪旧消息 | 用工具更新用户名到 state |

### 🔑 根本区别

- **Middleware**（wrap + node）= **你**根据写好的规则决定发生什么，LLM 根本不知道 middleware 存在
- **工具调用** = **LLM** 决定发生什么，它根据对话自己选择要不要调用

换个说法：
- Middleware = **护栏和管道**（LLM 看不见）
- 工具 = **能力**（LLM 看得见，它自己决定什么时候用）

### 📐 该用哪个？

| 我想要... | 用 | 为什么 |
|---|---|---|
| 按用户等级切模型 | 🔀 `wrap_model_call` | 开发者决策，LLM 不该知道 |
| 按角色过滤工具 | 🔀 `wrap_model_call` | 安全策略，不是 LLM 的选择 |
| 按用户生成提示词 | 🔀 `dynamic_prompt` | 提示词由开发者控制 |
| 裁剪旧消息 | 🪝 `@before_agent` | state 管理，LLM 看不到这层 |
| 总结对话 | 🪝 `SummarizationMiddleware` | state 管理，LLM 看不到这层 |
| 让 agent 读用户偏好 | 🔧 工具调用 | LLM 决定什么时候需要这个信息 |
| 让 agent 更新数据库 | 🔧 工具调用 | LLM 决定什么时候要写 |
| 让 agent 搜网页 | 🔧 工具调用 | LLM 决定什么时候要搜 |

### 🎯 本 notebook 要做的事

用这三种方式分别实现**同一个功能**——一个用户画像系统，这样能直观看出它们到底有什么不同。

In [ ]:
from dotenv import load_dotenv

load_dotenv()

---

## 🔀 方式一：Wrap 风格 Middleware

### "agent 运行时，调整它能用的工具、提示词、模型"

**场景**：内部用户能访问数据库工具，外部用户只能用网页搜索。LLM 根本不知道工具被过滤了——它看到的就是工具变少了。

**关键**：这是**开发者**做的决定，不是 LLM。LLM 无法绕过它。

In [ ]:
from dataclasses import dataclass
from typing import Callable
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage


# ── Two tools: one public, one restricted ──

@tool
def web_search(query: str) -> str:
    """Search the public web for information."""
    return f"Web results for: {query}"


@tool
def internal_database(sql: str) -> str:
    """Query the internal company database. Restricted to internal users."""
    return f"DB result: 42 rows matched for '{sql}'"


# ── Context: who is the user? ──

@dataclass
class UserProfile:
    user_role: str = "external"   # "internal" or "external"
    user_name: str = "Guest"


# ── Wrap-style middleware: filter tools by role ──

@wrap_model_call
def filter_tools_by_role(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """External users can only use web_search."""
    role = request.runtime.context.user_role

    if role != "internal":
        # Remove internal_database from the tools the LLM sees
        safe_tools = [t for t in request.tools if t.name != "internal_database"]
        request = request.override(tools=safe_tools)
        print(f"  🔀 [Middleware] External user → tools filtered to: {[t.name for t in safe_tools]}")
    else:
        print(f"  🔀 [Middleware] Internal user → all tools available")

    return handler(request)


agent_wrap = create_agent(
    model="gpt-5-nano",
    tools=[web_search, internal_database],
    context_schema=UserProfile,
    middleware=[filter_tools_by_role],
)

In [ ]:
# External user asks about the database → agent can't use it, falls back to web
print("=== External User ===")
response = agent_wrap.invoke(
    {"messages": [HumanMessage(content="How many customers do we have?")]},
    context=UserProfile(user_role="external", user_name="Bob"),
)
print(response["messages"][-1].content)

In [ ]:
# Internal user asks the same question → agent CAN use the database
print("=== Internal User ===")
response = agent_wrap.invoke(
    {"messages": [HumanMessage(content="How many customers do we have?")]},
    context=UserProfile(user_role="internal", user_name="Alice"),
)
print(response["messages"][-1].content)

### 🔍 刚刚发生了什么？

middleware 在 LLM 看到工具列表**之前**，悄悄把 `internal_database` 从里面摘掉了。外部用户那次 LLM 调用压根不知道这个工具存在过。这对 LLM **完全不可见**——是开发者强制执行的策略。

---

## 🪝 方式二：Node 风格 Middleware

### "agent 运行时，调整它的 state"

**场景**：每次 LLM 调用之前，把用户名作为一条系统级提示注入对话。这改的是 `messages[]`——也就是 state——而不是模型或工具配置。

**关键**：这改变的是 **state**（LLM 读到的内容），不是**配置**（用哪个模型/工具）。

In [ ]:
from langchain.agents.middleware import before_agent
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from langchain.messages import SystemMessage
from typing import Any


@before_agent
def inject_user_context(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Add a system message with the user's name before each LLM call."""
    user_name = runtime.context.user_name
    
    # Check if we already injected (avoid duplicates across turns)
    for msg in state["messages"]:
        if isinstance(msg, SystemMessage) and "Current user:" in msg.content:
            return None  # already injected
    
    print(f"  🪝 [Node middleware] Injecting user context: {user_name}")
    return {
        "messages": [SystemMessage(content=f"Current user: {user_name}. Address them by name.")]
    }


agent_node = create_agent(
    model="gpt-5-nano",
    context_schema=UserProfile,
    middleware=[inject_user_context],
)

In [ ]:
response = agent_node.invoke(
    {"messages": [HumanMessage(content="What's the weather like today?")]},
    context=UserProfile(user_name="Alice"),
)
# The agent should address Alice by name
print(response["messages"][-1].content)

In [ ]:
response = agent_node.invoke(
    {"messages": [HumanMessage(content="What's the weather like today?")]},
    context=UserProfile(user_name="Carlos"),
)
# Now it should address Carlos
print(response["messages"][-1].content)

### 🔍 刚刚发生了什么？

middleware 通过往 `messages[]` 里加一条 `SystemMessage` 来**修改 state**。LLM 看到这条消息，并用它来个性化回复。和 wrap 风格不同，这不改变用的是哪个模型或工具——它改变的是**LLM 读到了什么**。

---

## 🔧 方式三：工具调用

### "agent 自己决定调整 state 或读取运行时上下文"

**场景**：agent 有一个工具可以从 context 里查用户名，还有一个工具能把偏好存进 state。**LLM 自己决定**什么时候调用这些工具——不是开发者说了算。

**关键**：LLM 掌握主动权。它看到工具描述，自己选择什么时候用。

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.messages import ToolMessage


# ── Custom state to track preferences ──

from langchain.agents import AgentState as BaseAgentState

class PreferenceState(BaseAgentState):
    favourite_color: str


# ── Tool: read user name from context ──

@tool
def get_user_name(runtime: ToolRuntime) -> str:
    """Get the current user's name."""
    name = runtime.context.user_name
    print(f"  🔧 [Tool] LLM called get_user_name → {name}")
    return name


# ── Tool: save preference to state ──

@tool
def save_favourite_color(color: str, runtime: ToolRuntime) -> Command:
    """Save the user's favourite color to the agent's memory."""
    print(f"  🔧 [Tool] LLM called save_favourite_color → {color}")
    return Command(
        update={
            "favourite_color": color,
            "messages": [
                ToolMessage(
                    content=f"Saved favourite color: {color}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


# ── Tool: read preference from state ──

@tool
def get_favourite_color(runtime: ToolRuntime) -> str:
    """Read the user's favourite color from memory."""
    color = runtime.state.get("favourite_color", "not set yet")
    print(f"  🔧 [Tool] LLM called get_favourite_color → {color}")
    return color


agent_tools = create_agent(
    model="gpt-5-nano",
    tools=[get_user_name, save_favourite_color, get_favourite_color],
    context_schema=UserProfile,
    state_schema=PreferenceState,
    checkpointer=InMemorySaver(),
)

In [ ]:
# The LLM decides to call get_user_name to learn who it's talking to
print("=== Turn 1: Who am I? ===")
response = agent_tools.invoke(
    {"messages": [HumanMessage(content="Hi! What's my name?")]},
    {"configurable": {"thread_id": "demo-1"}},
    context=UserProfile(user_name="Alice"),
)
print(response["messages"][-1].content)

In [ ]:
# The LLM decides to call save_favourite_color
print("\n=== Turn 2: Save a preference ===")
response = agent_tools.invoke(
    {"messages": [HumanMessage(content="My favourite color is teal")]},
    {"configurable": {"thread_id": "demo-1"}},
    context=UserProfile(user_name="Alice"),
)
print(response["messages"][-1].content)
print("\nState:", response.get("favourite_color", "(not set)"))

In [ ]:
# The LLM decides to call get_favourite_color to recall it
print("\n=== Turn 3: Recall the preference ===")
response = agent_tools.invoke(
    {"messages": [HumanMessage(content="What's my favourite color?")]},
    {"configurable": {"thread_id": "demo-1"}},
    context=UserProfile(user_name="Alice"),
)
print(response["messages"][-1].content)

### 🔍 刚刚发生了什么？

**LLM 自己决定**了什么时候调用每个工具。我们没有强制它——只是给了带清晰描述的工具，它自己选择了：
1. 被问"我叫什么名字"时调用 `get_user_name`
2. 用户说"我最喜欢的颜色是青色"时调用 `save_favourite_color`
3. 被要求回忆时调用 `get_favourite_color`

这和 middleware 本质不同——LLM 才是**驾驶员**。

---

## ⚖️ 三种方式并排对比

| 维度 | 🔀 Wrap 风格 | 🪝 Node 风格 | 🔧 工具调用 |
|---|---|---|---|
| **谁决定** | 开发者（你） | 开发者（你） | LLM（agent） |
| **LLM 知道吗** | ❌ 不知道——工具被悄悄过滤 | ❌ 不知道——消息被悄悄修改 | ✅ 知道——LLM 看得到并主动调用 |
| **改变什么** | 模型、工具、提示词 | state（消息、自定义字段） | state（通过 Command）或读取 context |
| **接收什么** | `ModelRequest` | `AgentState` + `Runtime` | `ToolRuntime` |
| **何时跑** | 每次 LLM/工具调用前后 | 图节点前后 | LLM 决定要用时 |
| **能被绕过吗** | ❌ 不能——总会跑 | ❌ 不能——总会跑 | ✅ 能——LLM 可能压根不调用 |

---

## 🧠 心智模型

把 agent 想象成坐在办公桌前工作的人：

- **🔀 Wrap 风格 middleware** = **IT 部门**悄悄配置他的电脑。他们决定装什么软件、能访问哪些网站。坐在桌前的人不知道什么被拦了。
- **🪝 Node 风格 middleware** = **办公室经理**在他开始工作前帮他整理收件箱。旧邮件被归档，重要的被标星。这个人看到的是干净的收件箱，但不知道是谁整理的。
- **🔧 工具调用** = 他**桌上的工具**——电话、计算器、文件柜。这个人**自己选择**什么时候拿起电话或打开柜子，没人强迫他。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**：三者可以叠加互补，但如果搞不清楚"这件事到底该不该让 LLM 知道"，很容易把该用 middleware 做的安全策略，错误地做成了一个可以被 LLM 绕过的工具（或反过来）。先问"这应该是 LLM 的决策权，还是开发者的控制权"，再选机制。

</div>

### 🎯 决策流程图

```
这件事该不该让 LLM 自己决定？
  ├── 是 → 用工具调用（@tool + ToolRuntime）
  └── 否 → 是不是在决定用哪个模型/工具/提示词？
              ├── 是 → 用 Wrap 风格（@wrap_model_call、@dynamic_prompt）
              └── 否 → 用 Node 风格（@before_agent、SummarizationMiddleware）
```

---

## 💡 能同时用吗？

当然可以。实际上，生产环境的 agent 通常三种都用：

```python
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_user_name, save_preference, web_search],  # 🔧 工具调用
    middleware=[
        filter_tools_by_role,      # 🔀 Wrap 风格：RBAC
        SummarizationMiddleware(), # 🪝 Node 风格：管理历史
        language_prompt,           # 🔀 Config 风格：动态提示词
    ],
    context_schema=UserProfile,
    state_schema=PreferenceState,
    checkpointer=InMemorySaver(),
)
```

它们能自然组合是因为作用在不同层面：
1. Wrap 风格配置这次 LLM 调用怎么发
2. Node 风格清理 state
3. 工具给 LLM 能力

互不冲突，各管各的。

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：先问"这件事该不该让 LLM 知道/决定"——要让 LLM 决定就用工具；不该让它知道、但是关于用哪个模型/工具/提示词就用 wrap 风格；不该让它知道、但是关于 state 管理就用 node 风格。三者天然能叠加，不冲突。

</div>